# Day 16: Project 1 — MLflow Tracking + Pytest Tests

**Dataset:** diabetes_clean.csv
**Models:** Logistic Regression, XGBoost, LightGBM (from Days 13-14)
**Goal:** MLflow experiment tracking + pytest tests for preprocessing pipeline

In [5]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import xgboost as xgb
import lightgbm as lgb
import joblib
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

# Set up MLflow
mlflow.set_tracking_uri("file:mlruns")
mlflow.set_experiment("hospital_readmission")

# Load data
df = pd.read_csv('data/diabetes_clean.csv')
exclude_cols = ['encounter_id', 'patient_nbr', 'diag_1', 'diag_2', 'diag_3', 
                'readmitted', 'readmitted_binary']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]
y = df['readmitted_binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Data loaded: {X_train.shape}, MLflow URI: {mlflow.get_tracking_uri()}")

2026/09/13 22:35:04 INFO mlflow.tracking.fluent: Experiment with name 'hospital_readmission' does not exist. Creating a new experiment.


Data loaded: (81412, 50), MLflow URI: file:mlruns


In [6]:
# 1. DEFINE PREPROCESSOR (shared across models)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Fit preprocessor
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Preprocessor fitted. Train shape: {X_train_processed.shape}")

Preprocessor fitted. Train shape: (81412, 268)


In [7]:
# 2. MLFLOW: LOGISTIC REGRESSION
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

with mlflow.start_run(run_name="logistic_regression_baseline"):
    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)
    
    # Train
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    lr.fit(X_train_processed, y_train)
    
    # Predict
    y_proba = lr.predict_proba(X_test_processed)[:, 1]
    y_pred = lr.predict(X_test_processed)
    
    # Metrics
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("f1", f1)
    
    # Log model
    mlflow.sklearn.log_model(lr, "model")
    
    print(f"LR logged: ROC-AUC={roc_auc:.4f}, PR-AUC={pr_auc:.4f}, F1={f1:.4f}")

2026/09/13 22:36:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LR logged: ROC-AUC=0.6518, PR-AUC=0.2050, F1=0.2621


In [8]:
# 3. MLFLOW: XGBOOST (tuned)
with mlflow.start_run(run_name="xgboost_tuned"):
    # Log parameters
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)
    mlflow.log_param("random_state", 42)
    
    # Train
    xgb_clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    )
    xgb_clf.fit(X_train_processed, y_train)
    
    # Predict
    y_proba = xgb_clf.predict_proba(X_test_processed)[:, 1]
    y_pred = xgb_clf.predict(X_test_processed)
    
    # Metrics
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("f1", f1)
    
    # Log model
    mlflow.xgboost.log_model(xgb_clf, "model")
    
    print(f"XGBoost logged: ROC-AUC={roc_auc:.4f}, PR-AUC={pr_auc:.4f}, F1={f1:.4f}")

2026/09/13 22:38:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost logged: ROC-AUC=0.6888, PR-AUC=0.2380, F1=0.2845


In [9]:
# 4. MLFLOW: LIGHTGBM (tuned - best model)
with mlflow.start_run(run_name="lightgbm_tuned_best"):
    # Log parameters
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 7)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("num_leaves", 31)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    
    # Train
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=7,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    lgb_clf.fit(X_train_processed, y_train)
    
    # Predict
    y_proba = lgb_clf.predict_proba(X_test_processed)[:, 1]
    y_pred = lgb_clf.predict(X_test_processed)
    
    # Metrics
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("f1", f1)
    
    # Log model
    mlflow.lightgbm.log_model(lgb_clf, "model")
    
    print(f"LightGBM logged: ROC-AUC={roc_auc:.4f}, PR-AUC={pr_auc:.4f}, F1={f1:.4f}")

c:\Users\Bakrim\anaconda3\envs\myenv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Bakrim\anaconda3\envs\myenv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/09/13 22:38:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LightGBM logged: ROC-AUC=0.6894, PR-AUC=0.2375, F1=0.2833


In [10]:
# 5. PYTEST TESTS FOR PREPROCESSING PIPELINE
# We'll write tests as functions, then run them with pytest

def test_no_missing_after_transform():
# Test that preprocessor output has no missing values
    X_test_proc = preprocessor.transform(X_test)
    if hasattr(X_test_proc, 'toarray'):
        X_test_proc = X_test_proc.toarray()
    assert not np.isnan(X_test_proc).any(), "Preprocessor output contains NaN values"
    print("✓ test_no_missing_after_transform passed")

def test_known_input_known_output():
    # Test specific known input produces expected output shape and no errors
    # Use first row of training data as known input
    known_input = X_train.iloc[:1]
    transformed = preprocessor.transform(known_input)
    
    # Check output shape matches expected (1 row, n features)
    expected_n_features = X_train_processed.shape[1]
    if hasattr(transformed, 'toarray'):
        transformed = transformed.toarray()
    assert transformed.shape == (1, expected_n_features), \
        f"Expected shape (1, {expected_n_features}), got {transformed.shape}"
    
    # Check no NaN in output
    assert not np.isnan(transformed).any(), "Known input produced NaN output"
    print("✓ test_known_input_known_output passed")

def test_feature_engineering_invariants():
    """Test that engineered features maintain expected invariants"""
    # Check prior_admissions >= 0
    assert (X_train['prior_admissions'] >= 0).all(), "prior_admissions has negative values"
    
    # Check med_change_count >= 0
    assert (X_train['med_change_count'] >= 0).all(), "med_change_count has negative values"
    
    # Check med_change_count <= number of drug columns (23)
    assert (X_train['med_change_count'] <= 23).all(), "med_change_count exceeds max possible"
    
    # Check los_category values are valid
    valid_los = ['Short (1-3d)', 'Medium (4-7d)', 'Long (8-14d)', 'Very Long (14+d)']
    assert X_train['los_category'].isin(valid_los).all(), "Invalid los_category values"
    
    # Check age_midpoint in reasonable range
    assert X_train['age_midpoint'].between(5, 95).all(), "age_midpoint out of range"
    
    print("✓ test_feature_engineering_invariants passed")

def test_pipeline_serialization():
    """Test that preprocessor can be saved and loaded correctly"""
    import joblib
    import tempfile
    import os
    
    with tempfile.NamedTemporaryFile(suffix='.joblib', delete=False) as f:
        temp_path = f.name
    
    try:
        # Save
        joblib.dump(preprocessor, temp_path)
        
        # Load
        loaded_preprocessor = joblib.load(temp_path)
        
        # Test loaded preprocessor produces same output
        test_input = X_train.iloc[:5]
        original_output = preprocessor.transform(test_input)
        loaded_output = loaded_preprocessor.transform(test_input)
        
        if hasattr(original_output, 'toarray'):
            original_output = original_output.toarray()
        if hasattr(loaded_output, 'toarray'):
            loaded_output = loaded_output.toarray()
        
        np.testing.assert_array_almost_equal(original_output, loaded_output, decimal=5)
        print("✓ test_pipeline_serialization passed")
    finally:
        if os.path.exists(temp_path):
            os.unlink(temp_path)

def test_class_balance_preserved():
    """Test that stratified split preserves class balance"""
    train_prop = y_train.mean()
    test_prop = y_test.mean()
    overall_prop = y.mean()
    
    # Both should be close to overall (within 0.02)
    assert abs(train_prop - overall_prop) < 0.02, f"Train prop {train_prop:.4f} vs overall {overall_prop:.4f}"
    assert abs(test_prop - overall_prop) < 0.02, f"Test prop {test_prop:.4f} vs overall {overall_prop:.4f}"
    print("✓ test_class_balance_preserved passed")

# Run all tests
print("Running tests...")
test_no_missing_after_transform()
test_known_input_known_output()
test_feature_engineering_invariants()
test_pipeline_serialization()
test_class_balance_preserved()
print("\nAll tests passed! ✓")

Running tests...
✓ test_no_missing_after_transform passed
✓ test_known_input_known_output passed
✓ test_feature_engineering_invariants passed
✓ test_pipeline_serialization passed
✓ test_class_balance_preserved passed

All tests passed! ✓


In [11]:
# 6. MLFLOW: VIEW EXPERIMENTS
# Print all runs for this experiment
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("hospital_readmission")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

print(f"\n=== MLflow Experiment: {experiment.name} ===")
print(f"Total runs: {len(runs)}")
print()

for run in runs:
    params = run.data.params
    metrics = run.data.metrics
    print(f"Run: {run.info.run_name}")
    print(f"  Model: {params.get('model_type', 'N/A')}")
    print(f"  ROC-AUC: {metrics.get('roc_auc', 'N/A'):.4f}")
    print(f"  PR-AUC: {metrics.get('pr_auc', 'N/A'):.4f}")
    print(f"  F1: {metrics.get('f1', 'N/A'):.4f}")
    print()


=== MLflow Experiment: hospital_readmission ===
Total runs: 3

Run: lightgbm_tuned_best
  Model: LightGBM
  ROC-AUC: 0.6894
  PR-AUC: 0.2375
  F1: 0.2833

Run: xgboost_tuned
  Model: XGBoost
  ROC-AUC: 0.6888
  PR-AUC: 0.2380
  F1: 0.2845

Run: logistic_regression_baseline
  Model: LogisticRegression
  ROC-AUC: 0.6518
  PR-AUC: 0.2050
  F1: 0.2621

